In [39]:
import pandas as pd
import numpy as np

# Charger le fichier avec le bon séparateur (point-virgule)
df = pd.read_csv("titanic3.csv", sep=';', engine='python')

df

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1.0,1.0,"Allen, Miss. Elisabeth Walton",female,29,0.0,0.0,24160,"211,3375",B5,S,2,NaN,"St Louis, MO"
1,1.0,1.0,"Allison, Master. Hudson Trevor",male,"0,9167",1.0,2.0,113781,"151,5500",C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1.0,0.0,"Allison, Miss. Helen Loraine",female,2,1.0,2.0,113781,"151,5500",C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1.0,0.0,"Allison, Mr. Hudson Joshua Creighton",male,30,1.0,2.0,113781,"151,5500",C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1.0,0.0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25,1.0,2.0,113781,"151,5500",C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1305,3.0,0.0,"Zabour, Miss. Thamine",female,NaN,1.0,0.0,2665,"14,4542",NaN,C,NaN,NaN,NaN
1306,3.0,0.0,"Zakarian, Mr. Mapriededer",male,"26,5",0.0,0.0,2656,"7,2250",NaN,C,NaN,304.0,NaN
1307,3.0,0.0,"Zakarian, Mr. Ortin",male,27,0.0,0.0,2670,"7,2250",NaN,C,NaN,NaN,NaN
1308,3.0,0.0,"Zimmerman, Mr. Leo",male,29,0.0,0.0,315082,"7,8750",NaN,S,NaN,NaN,NaN


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1310 entries, 0 to 1309
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   pclass     1309 non-null   float64
 1   survived   1309 non-null   float64
 2   name       1309 non-null   object 
 3   sex        1309 non-null   object 
 4   age        1046 non-null   object 
 5   sibsp      1309 non-null   float64
 6   parch      1309 non-null   float64
 7   ticket     1309 non-null   object 
 8   fare       1308 non-null   object 
 9   cabin      295 non-null    object 
 10  embarked   1307 non-null   object 
 11  boat       486 non-null    object 
 12  body       121 non-null    float64
 13  home.dest  745 non-null    object 
dtypes: float64(5), object(9)
memory usage: 143.4+ KB


In [41]:

# Nettoyage : convertir les colonnes 'age' et 'fare' en float (virgules → points)
df['age'] = df['age'].replace(',', '.', regex=True).astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)
df['fare'] = df['fare'].replace(',', '.', regex=True).astype(str).str.extract(r'(\d+\.?\d*)')[0].astype(float)


# Vérification des valeurs non conformes
Ce bloc effectue des contrôles simples pour vous aider à repérer rapidement des données problématiques:

- Valeurs manquantes par colonne (utile pour décider d’une imputation ou d’un filtrage).
- Vérifications de base sur les colonnes numériques `age` et `fare` (valeurs manquantes, négatives, ou hors bornes raisonnables).
- Contrôles de cohérence pour quelques colonnes catégorielles si elles existent (`sex`, `pclass`, `embarked`).
- Détection de doublons (lignes entières et tickets répétés si la colonne `ticket` est présente).
- Repérage de valeurs extrêmes via la méthode IQR sur `age` et `fare`.


In [25]:
# Contrôles de qualité pour repérer les valeurs non conformes

# 1) Valeurs manquantes par colonne
missing = df.isna().sum().sort_values(ascending=False)
print("Valeurs manquantes par colonne:\n", missing)

# 2) Colonnes numériques: vérifications simples
# Age: NaN, négatif, > 100 (borne indicative)
mask_age_nan = df['age'].isna()
mask_age_neg = df['age'] < 0
mask_age_high = df['age'] > 100
print(f"\nAge non conforme: NaN={mask_age_nan.sum()}, négatives={mask_age_neg.sum()}, >100={mask_age_high.sum()}")
print(df.loc[mask_age_nan | mask_age_neg | mask_age_high, ['age']].head(10))

# Fare: NaN, négatif
mask_fare_nan = df['fare'].isna()
mask_fare_neg = df['fare'] < 0
print(f"\nFare non conforme: NaN={mask_fare_nan.sum()}, négatives={mask_fare_neg.sum()}")
print(df.loc[mask_fare_nan | mask_fare_neg, ['fare']].head(10))

# 3) Contrôles catégoriels (si colonnes présentes)
def check_values(col, allowed):
    if col in df.columns:
        bad_mask = ~df[col].isin(allowed) & df[col].notna()
        print(f"\n{col}: valeurs hors liste ({allowed}) -> {bad_mask.sum()}")
        print(df.loc[bad_mask, [col]].head(10))

check_values('sex', ['male','female'])
check_values('pclass', [1,2,3])
check_values('embarked', ['C','Q','S'])

# 4) Duplicats
dup_rows = df.duplicated().sum()
print(f"\nLignes dupliquées: {dup_rows}")
if 'ticket' in df.columns:
    dup_tickets = df['ticket'].duplicated().sum()
    print(f"Tickets dupliqués: {dup_tickets}")

# 5) Détection de valeurs extrêmes via IQR (age, fare)
for col in ['age','fare']:
    if col in df.columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5*iqr
        upper = q3 + 1.5*iqr
        out_mask = (df[col] < lower) | (df[col] > upper)
        print(f"\n{col}: valeurs extrêmes (IQR) -> {out_mask.sum()} | bornes [{lower:.2f}, {upper:.2f}]")
        print(df.loc[out_mask, [col]].head(10))

# 6) Résumé rapide des colonnes numériques clés
print("\nRésumé numérique:\n", df[['age','fare']].describe())

Valeurs manquantes par colonne:
 body         1189
cabin        1015
boat          824
home.dest     565
age           264
embarked        3
fare            2
sibsp           1
name            1
survived        1
pclass          1
sex             1
parch           1
ticket          1
dtype: int64

Age non conforme: NaN=264, négatives=0, >100=0
     age
15   NaN
37   NaN
40   NaN
46   NaN
59   NaN
69   NaN
70   NaN
74   NaN
80   NaN
106  NaN

Fare non conforme: NaN=2, négatives=0
      fare
1225   NaN
1309   NaN

sex: valeurs hors liste (['male', 'female']) -> 0
Empty DataFrame
Columns: [sex]
Index: []

pclass: valeurs hors liste ([1, 2, 3]) -> 0
Empty DataFrame
Columns: [pclass]
Index: []

embarked: valeurs hors liste (['C', 'Q', 'S']) -> 0
Empty DataFrame
Columns: [embarked]
Index: []

Lignes dupliquées: 0
Tickets dupliqués: 380

age: valeurs extrêmes (IQR) -> 9 | bornes [-6.00, 66.00]
       age
9     71.0
14    80.0
61    76.0
81    70.0
135   71.0
285   67.0
506   70.0
727   70.5
1

In [42]:
df.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1.0,1.0,"Allen, Miss. Elisabeth Walton",female,29.0000,0.0,0.0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1.0,1.0,"Allison, Master. Hudson Trevor",male,0.9167,1.0,2.0,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1.0,0.0,"Allison, Miss. Helen Loraine",female,2.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1.0,0.0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1.0,0.0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1.0,2.0,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


# 2. Masquage partiel — explication et code simple
Le *masquage partiel* consiste à garder seulement une partie visible d'une information sensible, pour réduire le risque d'identification directe.

Objectif:
- `ticket_masked`: ne garder visibles que les **3 derniers caractères** du ticket.
- `home_masked`: ne garder que la **dernière partie de l'adresse** (souvent la ville ou le dernier segment), ex.: "New York".

In [44]:
# 1) Masquer le ticket: convertir en texte puis prendre les 3 derniers caractères
df['ticket_masked'] = df['ticket'].astype(str).str[-3:]

# 2) Masquer home.dest: prendre la partie après la dernière virgule et enlever les espaces
#    - Si la valeur est manquante (NaN), on garde NaN
home_text = df['home.dest'].astype(str)
df['home_masked'] = home_text.where(df['home.dest'].notna()) \
    .str.split(',').str[-1].str.strip()

# 3) (Optionnel) Aperçu
df[['ticket', 'ticket_masked', 'home.dest', 'home_masked']].head()

# 4) Suppression des identifiants directs
df = df.drop(columns=['name', 'ticket', 'cabin', 'home.dest'])

In [46]:
df.head()

,pclass,survived,sex,age,sibsp,parch,fare,embarked,boat,body,ticket_masked,home_masked
0,1.0,1.0,female,29.0000,0.0,0.0,211.3375,S,2,NaN,160,MO
1,1.0,1.0,male,0.9167,1.0,2.0,151.5500,S,11,NaN,781,ON
2,1.0,0.0,female,2.0000,1.0,2.0,151.5500,S,NaN,NaN,781,ON
3,1.0,0.0,male,30.0000,1.0,2.0,151.5500,S,NaN,135.0,781,ON
4,1.0,0.0,female,25.0000,1.0,2.0,151.5500,S,NaN,NaN,781,ON


# 3. Agrégation (Binning) — Regrouper age en tranches anonymes
Le *binning* ou agrégation consiste à **regrouper des valeurs continues dans des catégories discrètes** (tranches). C'est utile pour:
- **Anonymiser**: au lieu de dire "25 ans", on dit "jeune adulte".
- **Simplifier**: réduire le nombre de valeurs distinctes.
- **Sécurité des données**: masquer la précision exacte tout en gardant l'information générale.

Nouvelle colonne `age_group` avec 5 catégories:
- 0–12 → enfant
- 13–18 → ado
- 19–35 → jeune adulte
- 36–60 → adulte
- 60+ → senior

In [47]:
# Définir les bornes et les étiquettes
segment = [0, 12, 18, 35, 60, float('inf')]  # Les frontières des tranches
label = ['enfant', 'ado', 'jeune adulte', 'adulte', 'senior']

# Utiliser pd.cut() pour regrouper automatiquement
df['age_group'] = pd.cut(df['age'], bins=segment, labels=label, right=True)

# Vérifier le résultat
df[['age', 'age_group']].head(10)

,age,age_group
0,29.0000,jeune adulte
1,0.9167,enfant
2,2.0000,enfant
3,30.0000,jeune adulte
4,25.0000,jeune adulte
5,48.0000,adulte
6,63.0000,senior
7,39.0000,adulte
8,53.0000,adulte
9,71.0000,senior


In [48]:

# 3. AGRÉGATION par groupe d'âge

def age_group(age):
    if pd.isna(age):
        return np.nan
    if age <= 12:
        return 'enfant'
    elif age <= 18:
        return 'ado'
    elif age <= 35:
        return 'jeune adulte'
    elif age <= 60:
        return 'adulte'
    else:
        return 'senior'

df['age_group'] = df['age'].apply(age_group)

df[['age', 'age_group']].head(10)

,age,age_group
0,29.0000,jeune adulte
1,0.9167,enfant
2,2.0000,enfant
3,30.0000,jeune adulte
4,25.0000,jeune adulte
5,48.0000,adulte
6,63.0000,senior
7,39.0000,adulte
8,53.0000,adulte
9,71.0000,senior


In [49]:

# 4. PERTURBATION aléatoire ±10% sur 'fare'

# np.random.seed(0)  # pour reproductibilité mais c'est global generatot=r c'est mieux que ce soit local
# Recommended local generator
np.random.default_rng(0)
perturbation = np.random.uniform(0.9, 1.1, size=len(df))
df['fare_perturbed'] = (df['fare'] * perturbation).round(2)
df[['fare', 'fare_perturbed']].head()

,fare,fare_perturbed
0,211.3375,220.29
1,151.5500,138.02
2,151.5500,148.40
3,151.5500,141.47
4,151.5500,161.31


In [50]:
# 1) Créer une clé stable pour la correspondance (l'index original)
df = df.reset_index(drop=False).rename(columns={'index': 'original_id'})
# afficher les 5 premières lignes pour vérification
df[['original_id']].head()


,original_id
0,0
1,1
2,2
3,3
4,4


In [51]:
# 2) Générer des pseudonymes uniques (par exemple, "ID0001", "ID0002", ...)
df['pseudonym_id'] = ['ID' + str(i).zfill(4) for i in range(1, len(df)+1)]
df[['original_id', 'pseudonym_id']].head()

,original_id,pseudonym_id
0,0,ID0001
1,1,ID0002
2,2,ID0003
3,3,ID0004
4,4,ID0005


In [52]:
# 3) Table de correspondance (séparée)
mapping_df = df[['pseudonym_id', 'original_id']].copy()
mapping_df.head()

,pseudonym_id,original_id
0,ID0001,0
1,ID0002,1
2,ID0003,2
3,ID0004,3
4,ID0005,4


In [53]:
# 4) (Optionnel) Dictionnaire de correspondance
mapping_dict = dict(zip(mapping_df['pseudonym_id'], mapping_df['original_id']))
mapping_dict

{'ID0001': 0,
 'ID0002': 1,
 'ID0003': 2,
 'ID0004': 3,
 'ID0005': 4,
 'ID0006': 5,
 'ID0007': 6,
 'ID0008': 7,
 'ID0009': 8,
 'ID0010': 9,
 'ID0011': 10,
 'ID0012': 11,
 'ID0013': 12,
 'ID0014': 13,
 'ID0015': 14,
 'ID0016': 15,
 'ID0017': 16,
 'ID0018': 17,
 'ID0019': 18,
 'ID0020': 19,
 'ID0021': 20,
 'ID0022': 21,
 'ID0023': 22,
 'ID0024': 23,
 'ID0025': 24,
 'ID0026': 25,
 'ID0027': 26,
 'ID0028': 27,
 'ID0029': 28,
 'ID0030': 29,
 'ID0031': 30,
 'ID0032': 31,
 'ID0033': 32,
 'ID0034': 33,
 'ID0035': 34,
 'ID0036': 35,
 'ID0037': 36,
 'ID0038': 37,
 'ID0039': 38,
 'ID0040': 39,
 'ID0041': 40,
 'ID0042': 41,
 'ID0043': 42,
 'ID0044': 43,
 'ID0045': 44,
 'ID0046': 45,
 'ID0047': 46,
 'ID0048': 47,
 'ID0049': 48,
 'ID0050': 49,
 'ID0051': 50,
 'ID0052': 51,
 'ID0053': 52,
 'ID0054': 53,
 'ID0055': 54,
 'ID0056': 55,
 'ID0057': 56,
 'ID0058': 57,
 'ID0059': 58,
 'ID0060': 59,
 'ID0061': 60,
 'ID0062': 61,
 'ID0063': 62,
 'ID0064': 63,
 'ID0065': 64,
 'ID0066': 65,
 'ID0067': 66,
 'ID0

In [ ]:
# 5) (Optionnel) Enregistrer la table de correspondance à part
mapping_df.to_csv('pseudonym_mapping.csv', index=False)

In [59]:
# 6) (Optionnel) Créer une version anonymisée en supprimant les identifiants directs
# Ajustez la liste selon les colonnes présentes dans votre DataFrame
cols_to_drop = [c for c in ['name', 'age', 'ticket', 'cabin', 'home.dest'] if c in df.columns]
df_anonymised = df.drop(columns=cols_to_drop)

In [60]:
# Export optionnel
df_anonymised.to_csv("titanic_anonymised.csv", index=False)

In [61]:
# Affichage final (optionnel)
df_anonymised.head()


,original_id,pclass,survived,sex,sibsp,parch,fare,embarked,boat,body,ticket_masked,home_masked,age_group,fare_perturbed,pseudonym_id
0,0,1.0,1.0,female,0.0,0.0,211.3375,S,2,NaN,160,MO,jeune adulte,220.29,ID0001
1,1,1.0,1.0,male,1.0,2.0,151.5500,S,11,NaN,781,ON,enfant,138.02,ID0002
2,2,1.0,0.0,female,1.0,2.0,151.5500,S,NaN,NaN,781,ON,enfant,148.40,ID0003
3,3,1.0,0.0,male,1.0,2.0,151.5500,S,NaN,135.0,781,ON,jeune adulte,141.47,ID0004
4,4,1.0,0.0,female,1.0,2.0,151.5500,S,NaN,NaN,781,ON,jeune adulte,161.31,ID0005



1. Quelle technique a le plus altéré l'utilisabilité des données ?

Agrégation (binning) de l'âge a probablement le plus altéré l'utilisabilité, car :

Perte de granularité : remplacer des valeurs continues (25 ans) par des catégories larges (19–35 : jeune adulte) perd toute la précision. Impossible de calculer un âge moyen exact, de faire des régressions linéaires précises, ou de détecter des tendances fines.
Biais d'analyse : les corrélations avec d'autres variables continues (ex. tarif) deviennent moins détectables ; les intervalles larges cachent des variations intra-groupe.
Irréversibilité : contrairement à la perturbation (où l'on garde une valeur numérique proche), le binning transforme l'information en catégorie discrète ; on ne peut pas récupérer l'âge original.
Comparaison avec les autres techniques :

Technique	Impact sur utilisabilité	Raison
Masquage partiel (ticket, home.dest)	Faible	Ces colonnes sont souvent peu utilisées pour l'analyse statistique ; garder les 3 derniers caractères ou la ville suffit généralement.
Perturbation ±10% (fare)	Modéré	Garde une valeur numérique proche ; moyennes et distributions restent valables, mais légèrement bruitées. Corrélations restent détectables.
Agrégation/Binning (age)	Élevé	Perte totale de la précision continue ; limite fortement les analyses quantitatives.
Pseudonymisation (P0001…)	Minimal	Remplace un identifiant par un autre ; aucune perte d'information sur les autres variables.
2. Si vous deviez publier ce jeu anonymisé, quelles colonnes garderiez-vous ? Pourquoi ?

Colonnes à GARDER (utiles et à faible risque) :

Colonne	Raison
pseudonym_id	Identifiant anonyme pour lier les lignes sans révéler l'identité.
pclass	Classe socio-économique (1, 2, 3) : utile pour analyses, non identifiante seule.
sex	Genre (male/female) : variable clé pour analyses de survie, faible risque seul.
age_group	Tranche d'âge agrégée (enfant, ado, etc.) : masque la précision, reste exploitable.
sibsp	Nombre de frères/sœurs/conjoints à bord : utile, peu identifiant seul.
parch	Nombre de parents/enfants à bord : idem.
fare_perturbed	Tarif bruité ±10% : conserve l'utilité pour analyses économiques, masque la valeur exacte.
embarked	Port d'embarquement (C, Q, S) : géographique général, peu risqué seul.
survived	Variable cible (0/1) : essentielle pour analyses de survie.
Colonnes à SUPPRIMER (trop identifiantes) :

Colonne	Raison
name	Identifiant direct évident.
ticket	Identifiant unique ; même les 3 derniers caractères peuvent être recoupés.
cabin	Localisation précise à bord, souvent vide mais potentiellement identifiante.
home.dest	Adresse personnelle ; même la ville seule peut être recoupée avec d'autres infos.
age	Âge exact : trop précis, remplacé par age_group.
fare	Tarif exact : remplacé par fare_perturbed.
original_id	Clé vers les données originales ; doit rester strictement confidentielle.
Dataset publié suggéré :
